# Assignment 2: Autonomous Mumbai Trip Planning Agent

**Topic:** Comparing CoT, ToT, and ReAct strategies  
**Task:** Plan a 3-day winter trip from Delhi to Mumbai for 3 people within ₹40,000.

## Goal
Build and compare three LLM-based planning strategies:
1. Chain-of-Thought (CoT)
2. Tree-of-Thought (ToT)
3. ReAct (Reason + Act)

## Constraints
- Starting city: Delhi
- Destination: Mumbai
- Duration: 3 days / 2 nights
- Travellers: 3
- Total budget: ₹40,000
- Season: Winter
- Budget includes travel, accommodation, food, local transport, sightseeing, and buffer

## Mid-task disruption
The budget is reduced from ₹40,000 to ₹32,000.  
Each strategy must revise its plan and we will compare its ability to adapt.

In [3]:
# Core Python imports
import os
import json
import time
from datetime import datetime

# Trip-planning task specification
TRIP_CONFIG = {
    "origin": "Delhi",
    "destination": "Mumbai",
    "travel_season": "Winter",
    "travel_dates": "To be selected",
    "travellers": 3,
    "days": 3,
    "nights": 2,
    "total_budget_inr": 40000,
    "budget_includes": [
        "return intercity travel",
        "accommodation",
        "meals",
        "local transport",
        "sightseeing",
        "emergency buffer"
    ],
    "preferences": [
        "budget-friendly",
        "realistic travel times",
        "winter-suitable activities",
        "safe and practical itinerary"
    ],
    "disruption": {
        "type": "budget_reduction",
        "new_budget_inr": 32000,
        "description": "The trip budget is reduced after the first itinerary is created."
    }
}

print(json.dumps(TRIP_CONFIG, indent=2))

{
  "origin": "Delhi",
  "destination": "Mumbai",
  "travel_season": "Winter",
  "travel_dates": "To be selected",
  "travellers": 3,
  "days": 3,
  "nights": 2,
  "total_budget_inr": 40000,
  "budget_includes": [
    "return intercity travel",
    "accommodation",
    "meals",
    "local transport",
    "sightseeing",
    "emergency buffer"
  ],
  "preferences": [
    "budget-friendly",
    "realistic travel times",
    "winter-suitable activities",
    "safe and practical itinerary"
  ],
  "disruption": {
    "type": "budget_reduction",
    "new_budget_inr": 32000,
    "description": "The trip budget is reduced after the first itinerary is created."
  }
}


In [4]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print("Cohere API key loaded:", COHERE_API_KEY is not None)

Cohere API key loaded: True


In [5]:
import cohere

co = cohere.ClientV2(api_key=COHERE_API_KEY)

In [6]:
response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {"role": "user", "content": "Hello! Give me a one-line introduction."}
    ]
)

print(response.message.content)

[ThinkingAssistantMessageResponseContentItem(type='thinking', thinking='The user asks: "Hello! Give me a one-line introduction." So we need to respond with a one-line introduction. Probably something like "I\'m Command, a Cohere-powered AI assistant, ready to help you with your queries." Or similar. Keep it one line. Provide a concise intro.'), TextAssistantMessageResponseContentItem(type='text', text="I'm Command, an AI assistant developed by Cohere, here to help you with information and tasks.")]


## Shared Agent Utilities

All strategies use the same trip constraints, model, and output requirements.
Only the reasoning strategy changes.

In [7]:
def build_trip_prompt(trip_config):
    """Creates the identical task description used by every strategy."""
    
    return f"""
You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: {trip_config["origin"]}
- Destination: {trip_config["destination"]}
- Travellers: {trip_config["travellers"]}
- Duration: {trip_config["days"]} days / {trip_config["nights"]} nights
- Maximum total budget: INR {trip_config["total_budget_inr"]}
- Budget includes: {", ".join(trip_config["budget_includes"])}
- Preferences: {", ".join(trip_config["preferences"])}

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.
"""

BASE_TRIP_PROMPT = build_trip_prompt(TRIP_CONFIG)

print(BASE_TRIP_PROMPT)


You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: Delhi
- Destination: Mumbai
- Travellers: 3
- Duration: 3 days / 2 nights
- Maximum total budget: INR 40000
- Budget includes: return intercity travel, accommodation, meals, local transport, sightseeing, emergency buffer
- Preferences: budget-friendly, realistic travel times, winter-suitable activities, safe and practical itinerary

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.



In [8]:
def extract_response_text(response):
    """Extracts only the final visible answer, excluding thinking content."""
    
    text_parts = []
    
    for item in response.message.content:
        if hasattr(item, "text"):
            text_parts.append(item.text)
    
    return "\n".join(text_parts)


def run_base_agent(system_prompt, user_prompt):
    """
    Sends a request to Command A+ and records observable performance data.
    This function will be reused by CoT, ToT, and ReAct.
    """
    
    start_time = time.time()
    
    response = co.chat(
        model="command-a-plus-05-2026",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    elapsed_seconds = round(time.time() - start_time, 2)
    final_answer = extract_response_text(response)
    
    return {
        "final_answer": final_answer,
        "execution_time_seconds": elapsed_seconds,
        "raw_response": response
    }

In [9]:
SYSTEM_PROMPT = """
You are a helpful AI travel-planning assistant.
Follow the user's instructions carefully and provide a clear, practical answer.
"""

test_result = run_base_agent(
    system_prompt=SYSTEM_PROMPT,
    user_prompt="In one sentence, confirm that you can help plan the Mumbai trip."
)

print(test_result["final_answer"])
print(f"\nExecution time: {test_result['execution_time_seconds']} seconds")

Yes, I can help you plan your Mumbai trip.

Execution time: 0.9 seconds


## Strategy 1: Chain-of-Thought (CoT) Planning Agent

The CoT agent follows one structured planning path:
trip requirements → budget allocation → itinerary → constraint check.

It does not explore alternatives or call tools.

In [10]:
COT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a linear Chain-of-Thought strategy.

Work through the task in one structured path:
1. Identify trip requirements
2. Allocate the available budget
3. Create a practical day-wise itinerary
4. Check the final budget and constraints

Do not explore multiple competing plans.
Do not use external tools or claim to have searched the web.
If a price is uncertain, make a reasonable estimate and label it as an assumption.
"""


def run_cot_agent(trip_config):
    cot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use a single linear planning approach.

Return your answer using exactly these headings:

## Planning Summary
Briefly explain the main planning stages you followed.

## Budget Allocation
Show estimated costs for return travel, accommodation, meals,
local transport, sightseeing, and buffer.

## Day-wise Itinerary
Give a practical plan for Day 1, Day 2, and Day 3.

## Constraint Check
Confirm the traveller count, duration, winter suitability,
and whether the total stays within INR {trip_config["total_budget_inr"]}.

## Assumptions
List any estimated prices or assumptions.
"""

    result = run_base_agent(
        system_prompt=COT_SYSTEM_PROMPT,
        user_prompt=cot_user_prompt
    )

    result["strategy"] = "CoT"
    result["planning_steps"] = 4

    return result

In [11]:
cot_initial_result = run_cot_agent(TRIP_CONFIG)

print(cot_initial_result["final_answer"])
print(f"\nStrategy: {cot_initial_result['strategy']}")
print(f"Planning steps: {cot_initial_result['planning_steps']}")
print(f"Execution time: {cot_initial_result['execution_time_seconds']} seconds")

## Planning Summary
I followed a single linear chain‑of‑thought:  
1. Identified the exact travel requirements (Delhi → Mumbai, 3 travellers, 3 days/2 nights, INR 40000 budget).  
2. Allocated the budget to the mandatory categories: return intercity travel, accommodation, meals, local transport, sightseeing, and a safety buffer.  
3. Constructed a practical day‑wise itinerary that uses realistic travel times, budget‑friendly options, and winter‑suitable activities in Mumbai.  
4. Verified that all constraints are met and the total stays well under the INR 40000 ceiling.

## Budget Allocation
| Category | Estimated Cost (INR) | Notes |
|----------|----------------------|-------|
| Return intercity travel (Delhi ↔ Mumbai) | **12,000** | 3 × AC‑3 tier Duronto tickets, INR 2,000 each way (assumed). |
| Accommodation (2 nights) | **5,000** | Budget hotel / 2‑BHK Airbnb, INR 2,500 per night for 3 people. |
| Meals (breakfast, lunch, dinner) | **4,000** | INR 150 per meal × 9 meals per travel

## Strategy 2: Tree-of-Thought (ToT) Planning Agent

The ToT agent creates multiple candidate itineraries, evaluates them against
the same constraints, and selects the strongest plan.

In [12]:
TOT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a Tree-of-Thought strategy.

Generate three distinct candidate trip plans before selecting one:
1. Budget-first plan
2. Balanced plan
3. Comfort-first plan

Evaluate each plan for budget compliance, practicality, travel time,
and winter suitability. Then choose the best plan.

Do not use external tools or claim to have searched the web.
Clearly label uncertain prices as estimates.
"""


def run_tot_agent(trip_config):
    tot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Create exactly three candidate Mumbai-trip plans:

- Branch A: lowest-cost / budget-first
- Branch B: balanced cost and experience
- Branch C: more comfort, while trying to stay within budget

Return your response using exactly these headings:

## Candidate Plans
Describe Branch A, Branch B, and Branch C with their estimated total costs.

## Branch Evaluation
Compare all three branches using:
- budget compliance
- practical travel time
- winter suitability
- overall trip experience

## Selected Plan
Choose the best branch and explain why.

## Final Day-wise Itinerary
Give the selected Day 1, Day 2, and Day 3 plan.

## Final Budget Breakdown
Show all major costs and the total.

## Assumptions
List estimated or uncertain prices.
"""

    result = run_base_agent(
        system_prompt=TOT_SYSTEM_PROMPT,
        user_prompt=tot_user_prompt
    )

    result["strategy"] = "ToT"
    result["candidate_branches"] = 3
    result["planning_steps"] = 6

    return result

In [13]:
tot_initial_result = run_tot_agent(TRIP_CONFIG)

print(tot_initial_result["final_answer"])
print(f"\nStrategy: {tot_initial_result['strategy']}")
print(f"Candidate branches: {tot_initial_result['candidate_branches']}")
print(f"Execution time: {tot_initial_result['execution_time_seconds']} seconds")

## Candidate Plans
**Branch A – Lowest‑Cost / Budget‑First**  
- Intercity travel: Sleeper‑class train (Delhi‑Mumbai round‑trip, 3 pax) – ≈ INR 9,000  
- Accommodation: Budget guesthouse, 2 nights, triple room – ≈ INR 1,600  
- Meals: Street‑food meals, 9 meals @ INR 100 each – ≈ INR 900  
- Local transport: Public bus/auto, 3 days – ≈ INR 300  
- Sightseeing: Free sites (Marine Drive, Gateway of India) – ≈ INR 0  
- Emergency buffer: 5 % of subtotal – ≈ INR 800  
**Estimated total:** **≈ INR 12,600**  

**Branch B – Balanced Cost & Experience**  
- Intercity travel: AC‑3 tier train (Delhi‑Mumbai round‑trip, 3 pax) – ≈ INR 10,800  
- Accommodation: Mid‑range 3‑star hotel, 2 nights, triple occupancy – ≈ INR 5,000  
- Meals: Mid‑range restaurant meals, 9 meals @ INR 150 each – ≈ INR 1,350  
- Local transport: Metro + auto‑rickshaw, 3 days – ≈ INR 500  
- Sightseeing: Entry fees & guide (Gateway of India, CSMT, Elephanta Caves, Siddhivinayak) – ≈ INR 1,200  
- Emergency buffer: 10 % of su

In [14]:
import json
from tavily import TavilyClient


def web_search(query, max_results=5):
    """Searches the web and returns compact, useful result data."""
    
    tavily_api_key = os.getenv("TAVILY_API_KEY")

    if not tavily_api_key:
        raise ValueError("TAVILY_API_KEY is missing from the .env file.")

    tavily = TavilyClient(api_key=tavily_api_key)

    search_response = tavily.search(
        query=query,
        max_results=max_results,
        search_depth="basic"
    )

    results = []

    for item in search_response.get("results", []):
        results.append({
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": item.get("content", "")
        })

    return results


def calculate_trip_budget(
    return_travel_per_person,
    accommodation_total,
    meals_total,
    local_transport_total,
    sightseeing_total,
    buffer_total,
    travellers=3
):
    """Calculates a transparent trip budget."""
    
    total = (
        return_travel_per_person * travellers
        + accommodation_total
        + meals_total
        + local_transport_total
        + sightseeing_total
        + buffer_total
    )

    return [{
        "return_travel_per_person_inr": return_travel_per_person,
        "travellers": travellers,
        "return_travel_total_inr": return_travel_per_person * travellers,
        "accommodation_total_inr": accommodation_total,
        "meals_total_inr": meals_total,
        "local_transport_total_inr": local_transport_total,
        "sightseeing_total_inr": sightseeing_total,
        "buffer_total_inr": buffer_total,
        "grand_total_inr": total,
        "within_40000_budget": total <= 40000
    }]

In [15]:
REACT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Searches the live web for current travel, hotel, transport, attraction, or weather information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A focused travel-related web-search query."
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum results to return, between 1 and 5."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_trip_budget",
            "description": "Calculates the complete Mumbai trip budget in INR for all travellers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "return_travel_per_person": {"type": "number"},
                    "accommodation_total": {"type": "number"},
                    "meals_total": {"type": "number"},
                    "local_transport_total": {"type": "number"},
                    "sightseeing_total": {"type": "number"},
                    "buffer_total": {"type": "number"},
                    "travellers": {"type": "integer"}
                },
                "required": [
                    "return_travel_per_person",
                    "accommodation_total",
                    "meals_total",
                    "local_transport_total",
                    "sightseeing_total",
                    "buffer_total"
                ]
            }
        }
    }
]

FUNCTION_MAP = {
    "web_search": web_search,
    "calculate_trip_budget": calculate_trip_budget
}

In [16]:
REACT_SYSTEM_PROMPT = """
You are a ReAct travel-planning agent.

You must use tools when needed:
- Use web_search to collect current travel information.
- Use calculate_trip_budget before giving the final itinerary.

Work iteratively:
Reason about missing information, call an appropriate tool, inspect its
result, and revise the plan.

Do not invent web-search results or calculation results.
Use the final tool evidence in your answer.
"""


def run_react_agent(trip_config, max_iterations=4):
    react_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use live web search to research relevant Mumbai travel information.
Then use the budget calculator tool before presenting the final plan.

Because exact travel dates are not yet fixed, treat prices as estimates,
not bookable quotations.

Return these final headings after using the tools:

## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": react_user_prompt}
    ]

    tool_log = []
    start_time = time.time()
    final_answer = ""

    for _ in range(max_iterations):
        response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            tools=REACT_TOOLS,
            temperature=0
        )

        tool_calls = response.message.tool_calls

        if not tool_calls:
            final_answer = extract_response_text(response)
            break

        messages.append(response.message)

        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_arguments = json.loads(tool_call.function.arguments)

            tool_result = FUNCTION_MAP[tool_name](**tool_arguments)

            tool_log.append({
                "tool_name": tool_name,
                "arguments": tool_arguments,
                "result": tool_result
            })

            tool_content = [
                {
                    "type": "document",
                    "document": {"data": json.dumps(item)}
                }
                for item in tool_result
            ]

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_content
            })

    elapsed_seconds = round(time.time() - start_time, 2)

        # If the agent used all tool iterations, force one final answer
    # using the collected tool results.
    if not final_answer:
        messages.append({
            "role": "user",
            "content": """
You have enough tool evidence now.
Do not call any more tools.

Create the final Mumbai trip plan using the required headings:
## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""
        })

        final_response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            temperature=0
        )

        final_answer = extract_response_text(final_response)

    return {
        "strategy": "ReAct",
        "final_answer": final_answer,
        "tool_log": tool_log,
        "tool_calls_count": len(tool_log),
        "execution_time_seconds": elapsed_seconds
    }

In [17]:
load_dotenv(override=True)

if os.getenv("TAVILY_API_KEY"):
    print("Tavily API key loaded successfully.")
else:
    
    print("Tavily API key is still missing.")

Tavily API key loaded successfully.


In [19]:
react_initial_result = run_react_agent(TRIP_CONFIG)

print(react_initial_result["final_answer"])
print(f"\nTool calls made: {react_initial_result['tool_calls_count']}")
print(f"Execution time: {react_initial_result['execution_time_seconds']} seconds")

## Tool Evidence
Based on web searches, I found:
- Delhi to Mumbai flight options taking ~2 hours with budget airlines (IndiGo, SpiceJet) costing ₹2,300-₹8,000 per person
- Budget accommodation in Mumbai ranging from ₹300-₹800 for dorms to ₹700-₹1,200 for private rooms
- Local transport costs averaging ₹50-₹150 per day using Mumbai's local train network
- Food costs of ₹150-₹300 per meal at budget eateries and street food
- Free attractions like Gateway of India, Chhatrapati Shivaji Terminus, and Marine Drive
- Budget calculator confirmed total cost of ₹26,300 for 3 travelers

## Budget Breakdown
| Category | Cost (INR) | Details |
|----------|------------|---------|
| Return Travel | ₹10,500 | 3 people × ₹3,500 (budget flight round-trip) |
| Accommodation | ₹6,000 | 2 nights × ₹1,000/night × 3 people (private rooms) |
| Meals | ₹5,400 | 3 days × ₹600/day × 3 people (street food + budget restaurants) |
| Local Transport | ₹900 | 3 days × ₹100/day × 3 people (local trains + occasional a

## Dynamic Replanning Test

After the original ₹40,000 plan is generated, the total budget is reduced
to ₹32,000. Each strategy must create a revised feasible itinerary.

In [20]:
def build_trip_prompt(trip_config):
    """Creates the identical task description used by every strategy."""

    replanning_context = trip_config.get("replanning_context", "")

    return f"""
You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: {trip_config["origin"]}
- Destination: {trip_config["destination"]}
- Travellers: {trip_config["travellers"]}
- Duration: {trip_config["days"]} days / {trip_config["nights"]} nights
- Maximum total budget: INR {trip_config["total_budget_inr"]}
- Budget includes: {", ".join(trip_config["budget_includes"])}
- Preferences: {", ".join(trip_config["preferences"])}

{replanning_context}

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.
"""

In [21]:
import copy


def create_disrupted_config(trip_config, new_budget):
    revised_config = copy.deepcopy(trip_config)

    original_budget = revised_config["total_budget_inr"]
    revised_config["total_budget_inr"] = new_budget

    revised_config["replanning_context"] = f"""
This is a replanning request.

An earlier plan was created with a budget of INR {original_budget}.
The available budget has now been reduced to INR {new_budget}.

Revise the plan to remain feasible. Clearly explain what was reduced,
replaced, or removed to meet the new budget.
"""

    return revised_config


DISRUPTED_TRIP_CONFIG = create_disrupted_config(
    TRIP_CONFIG,
    new_budget=32000
)

print(json.dumps(DISRUPTED_TRIP_CONFIG, indent=2))

{
  "origin": "Delhi",
  "destination": "Mumbai",
  "travel_season": "Winter",
  "travel_dates": "To be selected",
  "travellers": 3,
  "days": 3,
  "nights": 2,
  "total_budget_inr": 32000,
  "budget_includes": [
    "return intercity travel",
    "accommodation",
    "meals",
    "local transport",
    "sightseeing",
    "emergency buffer"
  ],
  "preferences": [
    "budget-friendly",
    "realistic travel times",
    "winter-suitable activities",
    "safe and practical itinerary"
  ],
  "disruption": {
    "type": "budget_reduction",
    "new_budget_inr": 32000,
    "description": "The trip budget is reduced after the first itinerary is created."
  },
  "replanning_context": "\nThis is a replanning request.\n\nAn earlier plan was created with a budget of INR 40000.\nThe available budget has now been reduced to INR 32000.\n\nRevise the plan to remain feasible. Clearly explain what was reduced,\nreplaced, or removed to meet the new budget.\n"
}


In [22]:
cot_replanned_result = run_cot_agent(DISRUPTED_TRIP_CONFIG)
tot_replanned_result = run_tot_agent(DISRUPTED_TRIP_CONFIG)
react_replanned_result = run_react_agent(DISRUPTED_TRIP_CONFIG)

In [23]:
print("=== CoT Replanned Result ===")
print(cot_replanned_result["final_answer"])

=== CoT Replanned Result ===
## Planning Summary
I followed a single‑step approach:
1. Defined the core requirements (Delhi ↔ Mumbai, 3 travellers, 3 days/2 nights, INR 32 000 total).  
2. Chose the cheapest feasible intercity option (train) and a budget hotel that can accommodate three people.  
3. Estimated realistic costs for meals, local transport, and sightseeing based on typical prices in Mumbai during winter.  
4. Added a modest emergency buffer.  
5. Adjusted the previous higher‑cost plan by replacing flights with trains, downgrading accommodation, and limiting meals to local eateries, which brings the total well under the new budget.

## Budget Allocation
| Category | Cost (INR) | Notes |
|----------|------------|-------|
| Return Intercity Travel (Train) | 12 000 | 3 travellers × INR 2 000 (Delhi‑Mumbai) × 2 legs |
| Accommodation (2 nights) | 2 400 | Triple occupancy budget hotel @ INR 1 200/night |
| Meals (3 days) | 3 600 | INR 400 per person per day (breakfast, lunch, din

In [24]:
print("\n=== ToT Replanned Result ===")
print(tot_replanned_result["final_answer"])


=== ToT Replanned Result ===
## Candidate Plans
**Branch A – Lowest‑Cost / Budget‑First**  
- Intercity travel: Sleeper‑class train (Delhi ↔ Mumbai), ₹800 × 3 × 2 = ₹4 800  
- Accommodation: Budget hostel/dormitory, ₹600 /night × 2 = ₹1 200  
- Meals: Street‑food & basic eateries, ₹400 × 3 travellers × 3 days = ₹3 600  
- Local transport: Local train daily tickets, ₹50 × 3 × 3 = ₹450  
- Sightseeing: Free sites + minimal entry fees, ≈ ₹200 total  
- Emergency buffer (5 % of subtotal): ≈ ₹1 000  
**Estimated total:** **₹11 250**  

**Branch B – Balanced Cost & Experience**  
- Intercity travel: AC 3‑tier train, ₹1 800 × 3 × 2 = ₹10 800  
- Accommodation: Mid‑range hotel (triple occupancy, breakfast included), ₹1 200 /night × 2 = ₹2 400  
- Meals: Mix of restaurants & street food, ₹600 × 3 × 3 = ₹5 400  
- Local transport: Auto/rideshare as needed, ₹150 × 3 × 3 = ₹1 350  
- Sightseeing: Key attractions (Elephanta ferry + entry, Siddhivinayak donation), ≈ ₹500 total  
- Emergency buffer 

In [25]:
print("\n=== ReAct Replanned Result ===")
print(react_replanned_result["final_answer"])


=== ReAct Replanned Result ===
## Tool Evidence
Based on web searches, I found current Delhi-Mumbai train prices (sleeper class ₹605-₹1000 per person), Mumbai budget accommodation (₹600-800 per night), local transport costs, and sightseeing options. The budget calculator confirmed total costs of ₹17700, well within the ₹32000 limit.

## Budget Breakdown
| Category | Cost (INR) | Details |
|----------|------------|---------|
| Return Travel | ₹2,400 | 3 travelers × ₹800 each (sleeper class trains) |
| Accommodation | ₹1,400 | 2 nights × ₹700 average (budget hotel/guesthouse for 3) |
| Meals | ₹10,800 | 3 travelers × 3 days × ₹1,200/day (mixed street food & restaurants) |
| Local Transport | ₹600 | Mumbai local trains & occasional auto-rickshaw |
| Sightseeing | ₹1,500 | Elephanta Caves (₹500 pp), Bollywood tour (₹1,000 pp), entry fees |
| Emergency Buffer | ₹1,000 | 5.6% of total for contingencies |
| **TOTAL** | **₹17,700** | **Under budget by ₹14,300** |

## Day-wise Itinerary

**Day

## Evaluation Framework

Each initial and replanned itinerary is evaluated using the same rubric:

- Budget compliance
- Traveller and duration constraints
- Practicality of itinerary
- Overall plan quality
- Dynamic replanning success

In [28]:
import pandas as pd


EVALUATOR_SYSTEM_PROMPT = """
You are a strict evaluator for AI-generated travel itineraries.

Evaluate only the plan provided. Do not invent missing facts.
Return valid JSON only.
"""


def evaluate_plan(plan_text, budget_limit):
    evaluation_prompt = f"""
Evaluate the following travel plan.

Budget limit: INR {budget_limit}

TRAVEL PLAN:
{plan_text}

Return a JSON object with exactly these fields:

{{
  "estimated_total_inr": number or null,
  "within_budget": true or false,
  "has_three_day_itinerary": true or false,
  "supports_three_travellers": true or false,
  "practicality_score_out_of_10": number,
  "overall_quality_score_out_of_10": number,
  "short_justification": "short explanation"
}}

Scoring guide:
- 9-10: complete, realistic, budget-aware, and practical
- 6-8: mostly good but has minor gaps or unclear assumptions
- 0-5: unrealistic, incomplete, over-budget, or missing key constraints
"""

    response = co.chat(
        model="command-a-plus-05-2026",
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": evaluation_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    return json.loads(extract_response_text(response))

In [29]:
evaluation_results = {
    "CoT - Initial": evaluate_plan(
        cot_initial_result["final_answer"],
        budget_limit=40000
    ),
    "ToT - Initial": evaluate_plan(
        tot_initial_result["final_answer"],
        budget_limit=40000
    ),
    "ReAct - Initial": evaluate_plan(
        react_initial_result["final_answer"],
        budget_limit=40000
    ),
    "CoT - Replanned": evaluate_plan(
        cot_replanned_result["final_answer"],
        budget_limit=32000
    ),
    "ToT - Replanned": evaluate_plan(
        tot_replanned_result["final_answer"],
        budget_limit=32000
    ),
    "ReAct - Replanned": evaluate_plan(
        react_replanned_result["final_answer"],
        budget_limit=32000
    )
}

In [30]:
comparison_rows = [
    {
        "Strategy": "CoT",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": cot_initial_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": cot_initial_result["execution_time_seconds"],
        **evaluation_results["CoT - Initial"]
    },
    {
        "Strategy": "ToT",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": tot_initial_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": tot_initial_result["execution_time_seconds"],
        **evaluation_results["ToT - Initial"]
    },
    {
        "Strategy": "ReAct",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": "Tool-driven",
        "Tool Calls": react_initial_result["tool_calls_count"],
        "Execution Time (s)": react_initial_result["execution_time_seconds"],
        **evaluation_results["ReAct - Initial"]
    },
    {
        "Strategy": "CoT",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": cot_replanned_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": cot_replanned_result["execution_time_seconds"],
        **evaluation_results["CoT - Replanned"]
    },
    {
        "Strategy": "ToT",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": tot_replanned_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": tot_replanned_result["execution_time_seconds"],
        **evaluation_results["ToT - Replanned"]
    },
    {
        "Strategy": "ReAct",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": "Tool-driven",
        "Tool Calls": react_replanned_result["tool_calls_count"],
        "Execution Time (s)": react_replanned_result["execution_time_seconds"],
        **evaluation_results["ReAct - Replanned"]
    }
]

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,Strategy,Scenario,Reasoning / Planning Steps,Tool Calls,Execution Time (s),estimated_total_inr,within_budget,has_three_day_itinerary,supports_three_travellers,practicality_score_out_of_10,overall_quality_score_out_of_10,short_justification
0,CoT,Initial,4,0,12.00,25300,True,True,True,9,9,"Detailed, realistic itinerary meets all constr..."
1,ToT,Initial,6,0,22.88,20850,True,True,True,9,9,"Balanced plan stays well under INR 40,000, sup..."
2,ReAct,Initial,Tool-driven,4,17.15,26300,True,True,True,7,8,"Plan is detailed, stays well under budget, and..."
3,CoT,Replanned,4,0,7.49,24500,True,True,True,9,9,"Complete, realistic 3‑day plan for 3 traveller..."
4,ToT,Replanned,6,0,19.45,22450,True,True,True,9,9,"Branch B meets budget, supports three travelle..."
5,ReAct,Replanned,Tool-driven,4,15.87,17700,True,True,True,9,9,"Complete, realistic itinerary for three travel..."
